# 01.08 - Faster R-CNN ResNet50-FPN V2 practice

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** Bounded Faster R-CNN training and inference evidence.

This third Object Detection lesson reinforces the official Torchvision detector boundary without downloads. The model is deliberately constrained for 64×64 toy images so the exercise remains practical on CPU.

## Core Ideas

Faster R-CNN is a two-stage detector: the Region Proposal Network proposes candidate regions, then ROI heads classify and refine them. Torchvision expects a list of `[C,H,W]` float images and, during training, one target dictionary per image with `boxes` (`float32 [K,4]`) and `labels` (`int64 [K]`). `min_size`, `max_size`, and RPN proposal limits dominate runtime on tiny inputs.

In [ ]:
import time
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2

SEED = 1
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Exercise 01-A: Build a bounded official detector

Use no downloaded weights. Preserve 64×64 inputs and cap proposal counts for this one-object fixture.

**Return structure — `build_practice_detector`:** A Torchvision `FasterRCNN` module on `device`, with three output classes (background plus two foreground classes), transform size 64, training RPN limits 200/100, testing limits 100/50, and at most 20 final detections.

In [ ]:
# TODO 01-A
def build_practice_detector(num_classes=3, device=DEVICE):
    raise NotImplementedError("Complete Exercise 01-A")


# Smoke check: construct the bounded official model.
detector = build_practice_detector()
print(type(detector).__name__, detector.transform.min_size, detector.transform.max_size)

## Exercise 01-B: Prepare image and target lists

Make two deterministic images with one square each. Detection labels start at 1 because 0 is reserved for background.

**Return structure — `make_practice_batch`:** A tuple `(images, targets)`. `images` is a list of two CPU `float32` tensors `[3,64,64]`. `targets` is a list of two dictionaries containing `boxes` (`float32 [1,4]`) and `labels` (`int64 [1]`).

In [ ]:
# TODO 01-B
def make_practice_batch():
    raise NotImplementedError("Complete Exercise 01-B")


# Smoke check: inspect the library boundary.
practice_images, practice_targets = make_practice_batch()
print(len(practice_images), practice_images[0].shape, practice_targets[0])

## Exercise 01-C: Run one training update

Move every image and target tensor to the same device. Sum the four detector losses before backpropagation.

**Return structure — `train_detection_batch`:** A dictionary with Python-float keys `loss_classifier`, `loss_box_reg`, `loss_objectness`, `loss_rpn_box_reg`, `total`, and `runtime_seconds`. The supplied model is updated once.

In [ ]:
# TODO 01-C
def train_detection_batch(model, images, targets, learning_rate=0.001, device=DEVICE):
    raise NotImplementedError("Complete Exercise 01-C")


# Smoke check: one speed-bounded structural update.
training_record = train_detection_batch(detector, practice_images, practice_targets)
print("training record:", training_record)

## Exercise 01-D: Run filtered inference

Return one record per image and keep only predictions at or above the threshold.

**Return structure — `infer_detection_batch`:** A `list[dict]` with one row per image. Each row contains `image_index` (`int`), `boxes` (`list[list[float]]`), `labels` (`list[int]`), `scores` (`list[float]`), and `runtime_seconds` (`float`). All three prediction lists have equal length and at most 20 items.

In [ ]:
# TODO 01-D
def infer_detection_batch(model, images, confidence_threshold=0.2, device=DEVICE):
    raise NotImplementedError("Complete Exercise 01-D")


# Smoke check: summarize predictions for both images.
inference_rows = infer_detection_batch(detector, practice_images)
print("inference rows:", inference_rows)

## Test Cases

**Return structure — `run_day01_tests`:** Returns `None`; assertions and `Day 01 tests passed` communicate success.

In [ ]:
def run_day01_tests():
    assert detector.transform.min_size == (64,) and detector.transform.max_size == 64
    assert detector.rpn._pre_nms_top_n == {"training": 200, "testing": 100}
    assert detector.rpn._post_nms_top_n == {"training": 100, "testing": 50}
    assert len(practice_images) == len(practice_targets) == 2
    assert practice_images[0].shape == (3, 64, 64) and practice_images[0].dtype == torch.float32
    assert practice_targets[0]["boxes"].shape == (1, 4) and practice_targets[0]["labels"].dtype == torch.int64
    assert set(training_record) == {"loss_classifier", "loss_box_reg", "loss_objectness", "loss_rpn_box_reg", "total", "runtime_seconds"}
    assert training_record["total"] > 0 and training_record["runtime_seconds"] > 0
    assert len(inference_rows) == 2 and all(len(row["boxes"]) == len(row["labels"]) == len(row["scores"]) <= 20 for row in inference_rows)
    print("Day 01 tests passed")


run_day01_tests()

## Day 01 Checklist

- [ ] Explain the RPN and ROI-head stages.
- [ ] Build correctly typed target dictionaries.
- [ ] Keep 64×64 images from being resized to 800×800.
- [ ] Interpret all four training losses and inference records.
- [ ] Run the test cases.